# Chapter 11 Lab — NeMo Guardrails: Topical + Fact-Check Rails

**Estimated time:** 25 minutes · **Read-and-run.** We simulate the two rails (the real thing is
configured in **Colang**) and test an off-topic and a factually-risky prompt.

In [ ]:
colang_config = '''
define rail topical:
  allow topics: [generative ai, llm, nvidia, inference]
  block otherwise
define rail fact_check:
  on claim -> verify against knowledge base; refuse if unsupported
'''
print(colang_config)

In [ ]:
ALLOWED = {'generative','ai','llm','nvidia','inference','gpu','model','triton'}
KB_FACTS = {'nvidia nim is a set of inference microservices',
            'triton serves models with dynamic batching'}

def topical_rail(prompt):
    toks=set(prompt.lower().replace('?',' ').split())
    return bool(toks & ALLOWED)
def fact_check_rail(answer):
    a=answer.lower().strip('. ')
    return any(a in fact or fact in a for fact in KB_FACTS)

def guarded_bot(prompt, candidate_answer):
    if not topical_rail(prompt):
        return 'BLOCKED by topical rail (off-topic)'
    if not fact_check_rail(candidate_answer):
        return 'REFUSED by fact-check rail (claim not supported by KB)'
    return 'ALLOWED: ' + candidate_answer

In [ ]:
print(guarded_bot('What is the best pizza topping?', 'Pepperoni'))
print(guarded_bot('What is NVIDIA NIM?', 'NVIDIA NIM is a set of inference microservices'))
print(guarded_bot('What is NVIDIA NIM?', 'NVIDIA NIM trains models from scratch for free'))

## Key takeaway
Match the **guardrail type to the risk**: topical rails keep the bot in scope; fact-checking
rails curb hallucination; toxicity/bias rails handle safety. Choosing the right evaluation and
rail for the specific risk is what this domain tests.